# m06_l02_breaker

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Finds the code companion. On Google Colab it downloads it once.
import glob, io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level3/releases/latest/download/prodlab-companion.zip"
if not os.path.isdir("shopassist"):
    if os.path.isdir(os.path.join("..", "shopassist")):
        os.chdir("..")  # opened from the notebooks folder on your computer
    else:
        found = glob.glob("*-companion")  # kept: your edited answers stay
        if not found and "google.colab" in sys.modules:
            try:
                data = io.BytesIO(urllib.request.urlopen(URL).read())
            except OSError:  # network blocks the download: upload the zip
                from google.colab import files
                data = next(iter(files.upload()))
            zipfile.ZipFile(data).extractall(".")
            found = glob.glob("*-companion")
        if found:
            os.chdir(found[0])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Companion ready" if os.path.isdir("shopassist") else
      "Companion not found: see Lesson 0.1 of the book")


## Breaker

In [ ]:
from prodlab.perf import FakeClock
from prodlab.reliability import CircuitBreaker, FlakyProvider, serve

clock = FakeClock()
provider = FlakyProvider(clock, seed=2, fail_rate=0.0, slow_rate=0.0,
                         outage=(30, 70))     # down from 30 s to 70 s
breaker = CircuitBreaker(clock, threshold=3, cool_down_s=20)
rows = serve(provider, breaker, clock, n=60, every_s=2.0)
last = None
for r in rows:
    if r["state"] != last:                    # print state changes
        t, state = r["t"], r["state"]
        print(f"t={t:5.1f}s  breaker {state:<9} {r['outcome']}")
        last = r["state"]
print("provider calls:", provider.calls, "of", len(rows), "requests")
